# Notebook 03_02 — Feature Engineering: LDA

**Supervised** linear feature extraction — uses class labels `y_train` during fit (this is the
correct, supervisor-required use of LDA, unlike the previous unsupervised misuse).

**Hard constraint**: LDA can produce at most `n_classes - 1` discriminant components.
With 10 classes here, max components = 9. When `K > 9`, components are clipped to 9 and
the remaining columns are zero-padded so the feature matrix shape still matches `K`
(documented in the `actual_K` column of the results).

**Results saved incrementally to** `results/fe_lda_raw.csv` — resumable on crash.

In [1]:
import os, sys, subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

CODE_DIR = '/content/UAV' if IN_COLAB else os.environ.get('UAV_CODE_DIR', 'D:/UAV_')

if IN_COLAB and not os.path.isdir(os.path.join(CODE_DIR, '.git')):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/haribawngg/UAV.git', CODE_DIR], check=True)

sys.path.insert(0, os.path.join(CODE_DIR, 'notebook'))
from common import *
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

print('Imports OK')

CPU: 16 logical cores detected -> N_JOBS=-1 (RF/XGB/LGBM/KNN use all cores)
Imports OK


In [2]:
d = load_data()
X_train, X_val, X_test = d['X_train'], d['X_val'], d['X_test']
y_train = d['y_train']
N_CLASSES = d['n_classes']
LDA_MAX_COMPONENTS = N_CLASSES - 1

METHOD_NAME = 'LDA'
RESULTS_CSV = f'{RESULTS_DIR}fe_lda_raw.csv'

print(f'n_classes={N_CLASSES}  ->  LDA max components = {LDA_MAX_COMPONENTS}')

n_classes=10  ->  LDA max components = 9


In [3]:
def transform_fn(K):
    K_lda = min(K, LDA_MAX_COMPONENTS)
    lda = LDA(n_components=K_lda)
    Xtr = lda.fit_transform(X_train, y_train)   # SUPERVISED: uses labels
    Xva = lda.transform(X_val)
    Xte = lda.transform(X_test)

    if K_lda < K:
        pad = K - K_lda
        Xtr = np.hstack([Xtr, np.zeros((len(Xtr), pad))])
        Xva = np.hstack([Xva, np.zeros((len(Xva), pad))])
        Xte = np.hstack([Xte, np.zeros((len(Xte), pad))])
        print(f'  [LDA K={K}] clipped to {K_lda} components (= n_classes - 1), zero-padded to {K}')

    return Xtr, Xva, Xte, K_lda

In [4]:
run_experiment_grid(
    method_name=METHOD_NAME,
    transform_fn=transform_fn,
    d=d,
    results_csv_path=RESULTS_CSV
)

[LDA] K=4 seed=42 DT     F1=0.8906  train=0.1s  inf=0.0000ms/sample  (1/90)
[LDA] K=4 seed=42 RF     F1=0.8900  train=1.7s  inf=0.0028ms/sample  (2/90)
[LDA] K=4 seed=42 XGB    F1=0.8904  train=1.6s  inf=0.0009ms/sample  (3/90)
[LDA] K=4 seed=42 LGBM   F1=0.8904  train=1.0s  inf=0.0023ms/sample  (4/90)
[LDA] K=4 seed=42 KNN    F1=0.7574  train=0.1s  inf=0.0229ms/sample  (5/90)
[LDA] K=4 seed=42 MLP    F1=0.8791  train=22.0s  inf=0.0016ms/sample  (6/90)
[LDA] K=4 seed=52 DT     F1=0.8911  train=0.1s  inf=0.0000ms/sample  (7/90)
[LDA] K=4 seed=52 RF     F1=0.8904  train=1.7s  inf=0.0027ms/sample  (8/90)
[LDA] K=4 seed=52 XGB    F1=0.8905  train=1.5s  inf=0.0009ms/sample  (9/90)
[LDA] K=4 seed=52 LGBM   F1=0.8899  train=1.0s  inf=0.0024ms/sample  (10/90)
[LDA] K=4 seed=52 KNN    F1=0.7572  train=0.1s  inf=0.0197ms/sample  (11/90)
[LDA] K=4 seed=52 MLP    F1=0.8833  train=21.9s  inf=0.0013ms/sample  (12/90)
[LDA] K=4 seed=62 DT     F1=0.8923  train=0.1s  inf=0.0000ms/sample  (13/90)
[LDA] 

In [5]:
res_df = pd.read_csv(RESULTS_CSV)
print(res_df.groupby(['K', 'classifier'])[['f1', 'train_time_s', 'inference_ms']].agg(['mean', 'std']).to_string())
print('\nactual_K used per K:')
print(res_df.groupby('K')['actual_K'].first())

                     f1           train_time_s            inference_ms              
                   mean       std         mean        std         mean           std
K  classifier                                                                       
4  DT          0.891611  0.000722     0.134875   0.005977     0.000049  4.925654e-07
   KNN         0.757180  0.000237     0.077271   0.002005     0.019790  2.010392e-03
   LGBM        0.890632  0.000483     0.935360   0.032326     0.002333  9.184064e-05
   MLP         0.880624  0.003372    23.529189   5.445168     0.001285  1.734655e-04
   RF          0.891132  0.000904     1.655899   0.047180     0.002748  2.553282e-05
   XGB         0.891335  0.000812     1.517631   0.039151     0.000907  2.001513e-05
8  DT          0.891640  0.000749     0.251056   0.004337     0.000048  1.813171e-06
   KNN         0.754120  0.005536     0.132513   0.001448     0.023098  2.510985e-03
   LGBM        0.891139  0.000589     1.196857   0.053551     0.0